# mEC: maze tuning against open-field tuning, unit by unit

Reads run `f2d171c7b3d6` of `run_spatial.py --engine link --dataset mec`; it never opens a data_dic
(`RUN_MODE = 'load'`). Write-up: `SPATIAL_TUNING.md`, stage C sections dated 2026-10-12. bp02 has 0-1 HD
units per pair and is not interpreted. Mice are the replicates.

In [1]:
import os, sys, glob, pickle
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath('.'))
os.environ.setdefault('TSA_DATASET', 'mec')
import maze_of_link as ml
RUN_MODE = 'load'
RUN = '/ceph/behrens/adam_harris/Taskspace_abstraction_mEC/data/spatial/link/f2d171c7b3d6'
shards = {os.path.basename(f)[:-4]: pickle.load(open(f, 'rb')) for f in sorted(glob.glob(f'{RUN}/shards/*.pkl'))}
T = pickle.load(open(sorted(glob.glob(f'{RUN}/merged/link_merged*.pkl'))[-1], 'rb'))['tables']
pd.set_option('display.width', 250)
pd.DataFrame([dict(recday=rd, **{k: v for k, v in sh['reflection'].items() if k != 'ears'}) for rd, sh in shards.items()])

,recday,s,s_maze,s_arena,ear_signs
0,bp01_20240321_20240323,-1.0,-1.0,1.0,"{0: -1.0, 1: -1.0, 2: -1.0, 3: -1.0, 4: 1.0, 5..."
1,bp01_20240324_20240325,-1.0,-1.0,1.0,"{0: -1.0, 1: -1.0, 2: -1.0, 3: -1.0, 4: -1.0, ..."
2,bp01_20240402_20240403,-1.0,-1.0,1.0,"{0: -1.0, 1: -1.0, 2: -1.0, 3: -1.0, 4: 1.0, 5..."
3,bp02_20240319_20240321,-1.0,-1.0,1.0,"{0: nan, 1: -1.0, 2: -1.0, 3: -1.0, 4: -1.0, 5..."
4,bp02_20240324_20240325,-1.0,-1.0,1.0,"{0: -1.0, 1: -1.0, 2: -1.0, 3: -1.0, 4: -1.0, ..."
5,bp02_20240326_20240327,-1.0,-1.0,1.0,"{0: -1.0, 1: -1.0, 2: -1.0, 3: -1.0, 4: 1.0, 5..."
6,bp02_20240402_20240403,-1.0,-1.0,1.0,"{0: -1.0, 1: -1.0, 2: -1.0, 3: -1.0, 4: 1.0, 5..."
7,bp04_20240321_20240323,-1.0,-1.0,1.0,"{0: -1.0, 1: -1.0, 2: -1.0, 3: -1.0, 4: 1.0, 5..."
8,bp04_20240324_20240325,-1.0,-1.0,1.0,"{0: -1.0, 1: -1.0, 2: -1.0, 3: -1.0, 4: -1.0, ..."


## HD coherence across the two rigs, per mouse

Mean R over maze->arena pairs, against unit-shuffle null draw i averaged over the same pairs.

In [2]:
pd.DataFrame([dict(selection=key, **r) for key in ('hd_both', 'hd_open_field') for r in ml.mouse_pool(shards, 'pairs', key)])

,selection,mouse,n_items,n_null,real,null_mean,p
0,hd_both,bp01,21,10000,0.801753,0.253526,0.000100
1,hd_both,bp04,8,120,0.673773,0.373792,0.016529
2,hd_open_field,bp01,26,10000,0.600760,0.173723,0.000100
3,hd_open_field,bp04,15,120,0.559323,0.340185,0.016529


## The rotation per maze-arena pair

arena = -maze + phi (s = -1). Same-day pairs of bp01 and bp04 share phi.

In [3]:
hp = T['hd_pairs']
hp[(hp.selection == 'hd_both') & (hp.mouse != 'bp02') & hp.phi_deg.notna()][['recday', 'maze', 'arena', 'n', 'phi_deg', 'R', 'p', 'R_other_reflection']].round(3)

,recday,maze,arena,n,phi_deg,R,p,R_other_reflection
0,bp01_20240321_20240323,0,4,13,68.346,0.859,0.000,0.304
2,bp01_20240321_20240323,1,4,12,81.666,0.833,0.000,0.192
4,bp01_20240321_20240323,2,4,12,78.055,0.896,0.000,0.170
6,bp01_20240321_20240323,3,4,21,-116.739,0.767,0.000,0.091
8,bp01_20240321_20240323,5,4,13,71.424,0.683,0.002,0.199
10,bp01_20240321_20240323,7,4,16,72.195,0.832,0.000,0.069
12,bp01_20240321_20240323,9,4,8,63.268,0.933,0.000,0.144
14,bp01_20240324_20240325,0,8,18,37.995,0.691,0.000,0.214
16,bp01_20240324_20240325,1,8,12,36.322,0.913,0.000,0.256
18,bp01_20240324_20240325,2,8,12,38.047,0.709,0.002,0.213


## Within-maze flips and closure

In [4]:
mp = T['maze_pairs']
x = mp[(mp.mouse != 'bp02') & mp.phi_deg.notna()]
print(f'{len(x)} testable maze-maze pairs, {int(x.flip_flag.sum())} with |phi| > 45 deg')
display(x[x.flip_flag][['recday', 'maze1', 'maze2', 'n', 'phi_deg', 'R', 'p']].round(3))
T['closure'].round(1)

69 testable maze-maze pairs, 6 with |phi| > 45 deg


,recday,maze1,maze2,n,phi_deg,R,p
2,bp01_20240321_20240323,0,3,13,168.767,0.834,0.000
7,bp01_20240321_20240323,1,3,10,-176.834,0.795,0.000
11,bp01_20240321_20240323,2,3,9,179.847,0.760,0.003
15,bp01_20240321_20240323,3,5,11,-177.814,0.818,0.001
16,bp01_20240321_20240323,3,7,14,176.130,0.906,0.000
17,bp01_20240321_20240323,3,9,9,-173.878,0.796,0.001


,recday,mouse,kind,maze,a1,a2,expected_deg,observed_deg,error_deg
0,bp01_20240321_20240323,bp01,maze pair,0-1,4,4,-2.7,13.3,16.0
1,bp01_20240321_20240323,bp01,maze pair,0-2,4,4,-7.2,9.7,16.9
2,bp01_20240321_20240323,bp01,maze pair,0-3,4,4,168.8,174.9,6.1
3,bp01_20240321_20240323,bp01,maze pair,0-5,4,4,-12.9,3.1,16.0
4,bp01_20240321_20240323,bp01,maze pair,0-7,4,4,3.8,3.8,0.0
...,...,...,...,...,...,...,...,...,...
68,bp04_20240324_20240325,bp04,maze pair,3-5,8,8,-11.5,-32.7,-21.2
69,bp04_20240324_20240325,bp04,maze pair,3-6,8,8,1.7,2.7,1.0
70,bp04_20240324_20240325,bp04,maze pair,4-5,8,8,0.9,-1.7,-2.7
71,bp04_20240324_20240325,bp04,maze pair,4-6,8,8,2.6,33.7,31.0


## Class links, per mouse

Label shuffles within recday, pooled per mouse (null draw i averaged over recdays).

In [5]:
keys = ['maze_R_by_of_hd', 'edge_by_of_border', 'of_speed_by_maze_node', 'of_speed_by_maze_bridge', 'of_si_vs_maze_stab']
pd.DataFrame([dict(link=k, **r) for k in keys for r in ml.mouse_pool(shards, 'links', k)])

,link,mouse,n_items,n_null,real,null_mean,p
0,maze_R_by_of_hd,bp01,3,10000,0.002708,-0.000373,0.473653
1,maze_R_by_of_hd,bp04,2,10000,0.344751,0.007945,0.008299
2,edge_by_of_border,bp04,1,10000,0.651572,0.010653,0.285171
3,of_speed_by_maze_node,bp01,3,10000,0.228748,0.096169,0.172083
4,of_speed_by_maze_node,bp04,1,10000,0.611322,0.354036,0.289271
5,of_si_vs_maze_stab,bp01,3,10000,-0.019466,0.000220,0.584442
6,of_si_vs_maze_stab,bp04,2,10000,-0.249475,-0.000277,0.907109
